# 2.1 Converting MCQ to OSQ

# Set OpenAI API Key

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
hf_token = os.getenv('OPENAI_API_KEY')
if hf_token:
    os.environ['OPENAI_API_KEY'] = hf_token
    print("OPENAI_API_KEY loaded successfully")
else:
    raise ValueError("OPENAI_API_KEY not found in .env file")



OPENAI_API_KEY loaded successfully


# Define Prompts

In [11]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ],
    temperature=0.7  # <--- add temperature here
)

print(completion.choices[0].message.content)


Hello! How can I assist you today?


In [12]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain the importance of systems engineering."}
    ],
    temperature=0.7  # <--- add temperature here
)

print(completion.choices[0].message.content)


Systems engineering is a crucial discipline for several reasons:

1. Integration: Systems engineering ensures different components of a project or system work together seamlessly. It focuses on designing and managing complex systems over their life cycles ensuring that all different parts are integrated properly.

2. Problem-solving: Systems engineering provides a structured solution process that considers both the business and technical needs of all customers. This holistic view enables engineers to create a solution that not only solves the problem at hand but also works in the larger system.

3. Risk Management: It allows for early detection of possible risks and issues, enabling preventative action and ensuring that the system can meet its requirements even in unforeseen circumstances.

4. Cost Effective: By considering the entire life cycle of a system, from conception to decommissioning, systems engineering helps to reduce costs and prevent overruns in both time and resources.

5

In [ ]:
# pip install openai>=1.0.0 pandas
import json
import pandas as pd
from openai import OpenAI

client = OpenAI()

# Prompts
## Placeholder Classifier/Filter Prompt. Consider switching to prompt in proposal.

In [ ]:
# -----------------------------
# PROMPTS (no placeholders inside)
# -----------------------------
CLASSIFIER_PROMPT = """You are a classifier that decides if a multiple-choice question (MCQ) can be converted into an open-ended short-answer question (OSQ) without losing validity.

Decision rules (all must be true to be "Yes"):
1) The concept can be assessed without showing options.
2) The answer has a canonical core (definition, short derivation, numeric result with units, or specific rationale).
3) Removing options will not introduce major ambiguity or change the construct being measured.
4) The scope can be clearly stated in <2 sentences, and the expected answer fits within 1–6 sentences or a numeric expression.

Return strict JSON with fields:
{
  "decision": "Yes" | "No",
  "reason": "<1-2 sentence explanation referencing the rules>"
}
"""

## Placeholder Converter Prompt. Consider switching to prompt in proposal.

In [ ]:
CONVERTER_HEADER = """You convert one MCQ into a precise open-ended short-answer (OSQ).

Conversion requirements:
1) Remove all multiple-choice phrasing; no "which of the following", no options references.
2) State the task unambiguously so a knowledgeable person can answer without options.
3) Keep difficulty approximately the same; if the original expects a fact, definition, derivation, numeric value, or short rationale, require that.
4) If computation is involved, request units and show any assumptions the respondent must use (constants, formulas).
5) Produce a canonical reference answer and a concise rubric that can grade fairly (0–2, 0–3, or 0–5 scale). Include partial-credit guidance.
6) Avoid giving away the answer in the prompt. Keep the OSQ answerable in ~1–6 sentences or a numeric expression.

Return strict JSON with fields:
{
  "osq_prompt": "<final open-ended question users will see>",
  "expected_answer": "<concise canonical answer>",
  "rubric": [
     {"criterion": "<what to look for>", "points": <int>, "full_credit_conditions": "<bullet or sentence>", "partial_credit_conditions": "<bullet or sentence>"}
  ],
  "difficulty": "easy|medium|hard"
}
"""

In [ ]:
# -----------------------------
# Helper: call API and parse JSON
# -----------------------------
def chat_json(prompt: str,
              model: str = "gpt-4o-mini",
              temperature: float = 0.0,
              max_tokens: int = 600) -> dict:
    """Call Chat Completions with low temperature and parse JSON."""
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    text = resp.choices[0].message.content.strip()
    try:
        if text.startswith("```"):
            text = text.strip("`")
            text = text[text.find("{"):]
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise ValueError(f"Model did not return valid JSON:\n{text}")

# -----------------------------
# Step 1: classify suitability
# -----------------------------
def classify_mcq_suitability(df: pd.DataFrame,
                             question_col: str = "question",
                             model: str = "gpt-4o-mini") -> pd.DataFrame:
    decisions, reasons = [], []
    for q in df[question_col].tolist():
        prompt = f"{CLASSIFIER_PROMPT}\n\nMCQ:\n{q}"
        out = chat_json(prompt, model=model, temperature=0.0, max_tokens=200)
        decisions.append(out.get("decision", "No"))
        reasons.append(out.get("reason", ""))
    df = df.copy()
    df["osq_suitable"] = decisions
    df["osq_suitable_reason"] = reasons
    return df

# -----------------------------
# Step 2: convert MCQ → OSQ
# -----------------------------
def convert_mcq_row(row: pd.Series,
                    model: str = "gpt-4o-mini",
                    temperature: float = 0.2,
                    max_tokens: int = 800) -> dict:
    q = row.get("question", "")
    choices = row.get("choices", [])
    correct_choice = row.get("correct_choice", "")
    explanation = row.get("explanation", "") or "N/A"

    prompt = (
        f"{CONVERTER_HEADER}\n\n"
        f"Input MCQ:\n"
        f"- Stem: {q}\n"
        f"- Choices: {json.dumps(choices, ensure_ascii=False)}\n"
        f"- Correct choice: {correct_choice}\n"
        f"- Reference explanation (if any): {explanation}\n"
    )
    return chat_json(prompt, model=model, temperature=temperature, max_tokens=max_tokens)

def convert_mcqs_to_osq(df: pd.DataFrame,
                        model: str = "gpt-4o-mini",
                        only_if_suitable: bool = True) -> pd.DataFrame:
    df = df.copy()
    osq_prompt, expected, rubric, difficulty = [], [], [], []
    for _, row in df.iterrows():
        if only_if_suitable and str(row.get("osq_suitable", "No")) != "Yes":
            osq_prompt.append(None)
            expected.append(None)
            rubric.append(None)
            difficulty.append(None)
            continue
        out = convert_mcq_row(row, model=model)
        osq_prompt.append(out.get("osq_prompt"))
        expected.append(out.get("expected_answer"))
        rubric.append(out.get("rubric"))
        difficulty.append(out.get("difficulty"))
    df["osq_prompt"] = osq_prompt
    df["expected_answer"] = expected
    df["rubric"] = rubric
    df["difficulty"] = difficulty
    return df

# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    mcqs = pd.DataFrame([
        {
            "id": "Q1",
            "question": "Which lifecycle model best supports iterative risk reduction through continuous stakeholder feedback?",
            "choices": ["Waterfall", "Spiral", "V-Model", "Big-Bang"],
            "correct_choice": "Spiral",
            "explanation": "The Spiral model emphasizes iterative cycles with risk-driven prototyping and stakeholder input."
        },
        {
            "id": "Q2",
            "question": "What is the formula for Shannon capacity of an AWGN channel?",
            "choices": ["C=B log2(1+SNR)", "C=B ln(1+SNR)", "C=B log10(1+SNR)", "C=log2(1+SNR)"],
            "correct_choice": "C=B log2(1+SNR)",
            "explanation": "Capacity is C = B * log2(1 + SNR)."
        }
    ])

    # Step 1: classify
    mcqs = classify_mcq_suitability(mcqs, question_col="question")

    # Step 2: convert
    converted = convert_mcqs_to_osq(mcqs, only_if_suitable=True)

    print(converted[["id", "osq_suitable", "osq_prompt", "expected_answer", "difficulty"]].to_string())


   id osq_suitable                                                                                                      osq_prompt                                                                                   expected_answer difficulty
0  Q1          Yes  Describe the lifecycle model that emphasizes iterative risk reduction through continuous stakeholder feedback.  The Spiral model emphasizes iterative cycles with risk-driven prototyping and stakeholder input.     medium
1  Q2          Yes                                    What is the formula for calculating the Shannon capacity of an AWGN channel?                                                                             C = B * log2(1 + SNR)     medium


# Adapting for SysEngBench

both currently need to provide the actual MCQ choices. they are not populating. the confidence level is also not populating.

## Prompts Version 1

In [20]:
# === MCQ → OSQ in one Jupyter cell (self-contained) ===
# Prereqs: pip install openai>=1.0.0 pandas
# Make sure OPENAI_API_KEY is set in your environment before running.

import os, re, json
import pandas as pd
from typing import List, Dict, Any
from openai import OpenAI

# -----------------------------
# Config
# -----------------------------
# CSV_IN   = "/mnt/data/sysengbench.csv"            # input CSV
CSV_IN = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase1_prep\\sysengbench.csv"
CSV_OUT  = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase2_conversion\\sysengbench_converted.csv"  # output CSV
JSONL_OUT= "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase2_conversion\\sysengbench_converted.jsonl"
MODEL    = "gpt-4o-mini"                          # change if you like
SAMPLE_N = 5                                      # set >0 to test on small sample
SKIP_CLASSIFY = False                             # True = force convert all

# -----------------------------
# Client
# -----------------------------
client = OpenAI()  # relies on OPENAI_API_KEY

# -----------------------------
# Prompts (literal JSON; no .format())
# -----------------------------
CLASSIFIER_PROMPT = """You are a classifier that decides if a multiple-choice question (MCQ) can be converted into an open-ended short-answer question (OSQ) without losing validity.

Decision rules (all must be true to be "Yes"):
1) The concept can be assessed without showing options.
2) The answer has a canonical core (definition, short derivation, numeric result with units, or specific rationale).
3) Removing options will not introduce major ambiguity or change the construct being measured.
4) The scope can be clearly stated in <2 sentences, and the expected answer fits within 1–6 sentences or a numeric expression.

Return strict JSON with fields:
{
  "decision": "Yes" | "No",
  "reason": "<1-2 sentence explanation referencing the rules>"
}
"""

CONVERTER_HEADER = """You convert one MCQ into a precise open-ended short-answer (OSQ).

Conversion requirements:
1) Remove all multiple-choice phrasing; no "which of the following", no options references.
2) State the task unambiguously so a knowledgeable person can answer without options.
3) Keep difficulty approximately the same; if the original expects a fact, definition, derivation, numeric value, or short rationale, require that.
4) If computation is involved, request units and show any assumptions the respondent must use (constants, formulas).
5) Produce a canonical reference answer and a concise rubric that can grade fairly (0–2, 0–3, or 0–5 scale). Include partial-credit guidance.
6) Avoid giving away the answer in the prompt. Keep the OSQ answerable in ~1–6 sentences or a numeric expression.

Return strict JSON with fields:
{
  "osq_prompt": "<final open-ended question users will see>",
  "expected_answer": "<concise canonical answer>",
  "rubric": [
     {"criterion": "<what to look for>", "points": <int>, "full_credit_conditions": "<bullet or sentence>", "partial_credit_conditions": "<bullet or sentence>"}
  ],
  "difficulty": "easy|medium|hard"
}
"""

# -----------------------------
# Helpers (API + JSON parsing)
# -----------------------------
def chat_json(prompt: str,
              model: str = MODEL,
              temperature: float = 0.0,
              max_tokens: int = 600) -> Dict[str, Any]:
    """Call Chat Completions and parse a top-level JSON object robustly."""
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    text = resp.choices[0].message.content.strip()
    try:
        if text.startswith("```"):
            text = text.strip("`")
            idx = text.find("{")
            if idx >= 0:
                text = text[idx:]
        return json.loads(text)
    except Exception:
        start = text.find("{"); end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise ValueError(f"Model did not return valid JSON:\n{text}")

# -----------------------------
# CSV normalization utils
# -----------------------------
def coalesce_cols(df: pd.DataFrame, names: List[str], default=None):
    for n in names:
        if n in df.columns and df[n].notna().any():
            return df[n]
    return pd.Series([default] * len(df))

def parse_choices_field(val) -> List[str]:
    """Accepts JSON array, python list repr, or delimited strings."""
    if isinstance(val, list):
        return [str(x).strip() for x in val if str(x).strip()]
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    s = str(val).strip()
    if not s:
        return []
    # Try JSON list
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    # Try python literal list
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = eval(s, {"__builtins__": {}})
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
    # Fallback: split by common delimiters
    import re
    parts = re.split(r"\s*[;|,]\s*", s)
    return [p for p in parts if p]

def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce DF with: id, question, choices(list), correct_choice, explanation
    Tries common column names automatically.
    """
    df = df.copy()
    df["question"] = coalesce_cols(df, ["question", "prompt", "stem", "text", "mcq", "MCQ"], default="").astype(str)
    df["explanation"] = coalesce_cols(df, ["explanation", "solution", "rationale", "reasoning"], default="")
    df["correct_choice"] = coalesce_cols(df, ["correct_choice", "answer", "correct", "key", "label", "gold"], default="")

    # choices column or collect option-like columns
    if "choices" in df.columns:
        df["choices"] = df["choices"].apply(parse_choices_field)
    else:
        import re
        choice_like = [c for c in df.columns
                       if re.fullmatch(r"(choice_\d+|option_\d+|opt_\d+|A|B|C|D|E|F|G|H|option[A-H])", str(c), flags=re.IGNORECASE)]
        if choice_like:
            def row_to_choices(row):
                vals = []
                for c in choice_like:
                    v = row.get(c, None)
                    if v is not None and str(v).strip():
                        vals.append(str(v).strip())
                return vals
            df["choices"] = df.apply(row_to_choices, axis=1)
        else:
            df["choices"] = coalesce_cols(df, ["options", "answers", "candidates"], default="").apply(parse_choices_field)

    if "id" not in df.columns:
        df["id"] = coalesce_cols(df, ["qid", "uid", "index", "row_id"], default=None)
        if df["id"].isna().all():
            df["id"] = [f"Q{ix+1}" for ix in range(len(df))]

    df["choices"] = df["choices"].apply(lambda x: x if isinstance(x, list) else [])
    df["correct_choice"] = df["correct_choice"].astype(str)
    df["explanation"] = df["explanation"].astype(str)
    front = ["id", "question", "choices", "correct_choice", "explanation"]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]

# -----------------------------
# Pipeline (same modular funcs)
# -----------------------------
def classify_mcq_suitability(df: pd.DataFrame,
                             question_col: str = "question",
                             model: str = MODEL) -> pd.DataFrame:
    decisions, reasons = [], []
    for q in df[question_col].tolist():
        prompt = f"{CLASSIFIER_PROMPT}\n\nMCQ:\n{q}"
        out = chat_json(prompt, model=model, temperature=0.0, max_tokens=200)
        decisions.append(out.get("decision", "No"))
        reasons.append(out.get("reason", ""))
    df = df.copy()
    df["osq_suitable"] = decisions
    df["osq_suitable_reason"] = reasons
    return df

def convert_mcq_row(row: pd.Series,
                    model: str = MODEL,
                    temperature: float = 0.2,
                    max_tokens: int = 900) -> Dict[str, Any]:
    q = row.get("question", "")
    choices = row.get("choices", [])
    correct_choice = row.get("correct_choice", "")
    explanation = row.get("explanation", "") or "N/A"
    prompt = (
        f"{CONVERTER_HEADER}\n\n"
        f"Input MCQ:\n"
        f"- Stem: {q}\n"
        f"- Choices: {json.dumps(choices, ensure_ascii=False)}\n"
        f"- Correct choice: {correct_choice}\n"
        f"- Reference explanation (if any): {explanation}\n"
    )
    return chat_json(prompt, model=model, temperature=temperature, max_tokens=max_tokens)

def convert_mcqs_to_osq(df: pd.DataFrame,
                        model: str = MODEL,
                        only_if_suitable: bool = True) -> pd.DataFrame:
    df = df.copy()
    osq_prompt, expected, rubric, difficulty = [], [], [], []
    for _, row in df.iterrows():
        if only_if_suitable and str(row.get("osq_suitable", "No")) != "Yes":
            osq_prompt.append(None); expected.append(None); rubric.append(None); difficulty.append(None)
            continue
        out = convert_mcq_row(row, model=model)
        osq_prompt.append(out.get("osq_prompt"))
        expected.append(out.get("expected_answer"))
        rubric.append(out.get("rubric"))
        difficulty.append(out.get("difficulty"))
    df["osq_prompt"] = osq_prompt
    df["expected_answer"] = expected
    df["rubric"] = rubric
    df["difficulty"] = difficulty
    return df

# -----------------------------
# Run: load → normalize → (classify) → convert → save
# -----------------------------
df_raw = pd.read_csv(CSV_IN)
df = normalize_schema(df_raw)

if SAMPLE_N and SAMPLE_N > 0:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)

if SKIP_CLASSIFY:
    df["osq_suitable"] = "Yes"
    df["osq_suitable_reason"] = "Forced conversion (skip_classify)."
else:
    df = classify_mcq_suitability(df)

df_out = convert_mcqs_to_osq(df, only_if_suitable=not SKIP_CLASSIFY)

# Save files
df_out.to_csv(CSV_OUT, index=False)
with open(JSONL_OUT, "w", encoding="utf-8") as f:
    for _, row in df_out.iterrows():
        rec = {
            "id": row.get("id"),
            "question": row.get("question"),
            "choices": row.get("choices"),
            "correct_choice": row.get("correct_choice"),
            "explanation": row.get("explanation"),
            "osq_suitable": row.get("osq_suitable"),
            "osq_suitable_reason": row.get("osq_suitable_reason"),
            "osq_prompt": row.get("osq_prompt"),
            "expected_answer": row.get("expected_answer"),
            "rubric": row.get("rubric"),
            "difficulty": row.get("difficulty"),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Display a quick sample
print(f"Input rows: {len(df_raw)} | Processed rows: {len(df_out)}")
print("First few converted items:")
display(df_out[["id","osq_suitable","osq_prompt","expected_answer","difficulty"]].head(5))
print(f"\nSaved:\n  CSV  -> {CSV_OUT}\n  JSONL-> {JSONL_OUT}")


Input rows: 1144 | Processed rows: 5
First few converted items:


,id,osq_suitable,osq_prompt,expected_answer,difficulty
0,Q219,Yes,What is the focus of Lean Engineering in syste...,The focus of Lean Engineering in system lifecy...,medium
1,Q810,Yes,Describe the recommended approach for arrangin...,The recommended approach for arranging control...,medium
2,Q502,Yes,Explain the role of the Initial Capabilities D...,The Initial Capabilities Document (ICD) outlin...,medium
3,Q650,Yes,Describe the performance requirements for ship...,Shipboard equipment must demonstrate that it c...,medium
4,Q324,Yes,Describe how the interaction between an actor ...,"In SysML, the interaction between an actor and...",medium



Saved:
  CSV  -> C:\Users\rabel\Desktop\dissertation\src\phase2_conversion\sysengbench_converted.csv
  JSONL-> C:\Users\rabel\Desktop\dissertation\src\phase2_conversion\sysengbench_converted.jsonl


## Prompts Version 2

In [2]:
# === MCQ → OSQ in one Jupyter cell (self-contained) ===
# Prereqs: pip install openai>=1.0.0 pandas
# Make sure OPENAI_API_KEY is set in your environment before running.

import os, re, json
import pandas as pd
from typing import List, Dict, Any
from openai import OpenAI

# -----------------------------
# Config
# -----------------------------
# CSV_IN   = "/mnt/data/sysengbench.csv"            # input CSV
CSV_IN = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase1_prep\\sysengbench.csv"
CSV_OUT  = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase2_conversion\\sysengbench_converted.csv"  # output CSV
JSONL_OUT= "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase2_conversion\\sysengbench_converted.jsonl"
MODEL    = "gpt-4o-mini"                          # change if you like
SAMPLE_N = 5                                      # set >0 to test on small sample
SKIP_CLASSIFY = False                             # True = force convert all

# -----------------------------
# Client
# -----------------------------
client = OpenAI()  # relies on OPENAI_API_KEY

# -----------------------------
# Prompts (literal JSON; no .format())
# -----------------------------
CLASSIFIER_PROMPT = """You are a classifier that decides if a multiple-choice question (MCQ) can be converted into an open-ended short-answer question (OSQ) without losing validity.

Decision rules (all must be true to be "Yes"):
1) The concept can be assessed without showing options.
2) The answer has a canonical core (definition, short derivation, numeric result with units, or specific rationale).
3) Removing options will not introduce major ambiguity or change the construct being measured.
4) The scope can be clearly stated in <2 sentences, and the expected answer fits within 1–6 sentences or a numeric expression.
5) The question is suitable for conversion while maintaining intent and semantics.

Be sure to provide a confidence score from 1–10 indicating how confident you are in your decision (1 = very low, 10 = very high).

Return STRICT JSON ONLY with the following fields (no extra keys, no commentary):
{
  "decision": "Yes" | "No",
  "reason": "<1-2 sentence explanation referencing the rules>",
  "confidence": <integer 1-10>
}
"""

CONVERTER_HEADER = """You convert one MCQ into a precise open-ended short-answer (OSQ).

Conversion requirements:
1) Remove all multiple-choice phrasing; no "which of the following", no options references.
2) State the task unambiguously so a knowledgeable person can answer without options.
3) Keep difficulty approximately the same; if the original expects a fact, definition, derivation, numeric value, or short rationale, require that.
4) If computation is involved, request units and show any assumptions the respondent must use (constants, formulas).
5) Produce a canonical reference answer and a concise rubric that can grade fairly (0–5 scale). Include partial-credit guidance.
6) Avoid giving away the answer in the prompt. Keep the OSQ answerable in ~1–6 sentences or a numeric expression.

Return strict JSON with fields:
{
  "osq_prompt": "<final open-ended question users will see>",
  "expected_answer": "<concise canonical answer>",
  "rubric": [
     {"criterion": "<what to look for>", "points": <int>, "full_credit_conditions": "<bullet or sentence>", "partial_credit_conditions": "<bullet or sentence>"}
  ],
  "difficulty": "easy|medium|hard"
}
"""

# -----------------------------
# Helpers (API + JSON parsing)
# -----------------------------
def chat_json(prompt: str,
              model: str = MODEL,
              temperature: float = 0.0,
              max_tokens: int = 600) -> Dict[str, Any]:
    """Call Chat Completions and parse a top-level JSON object robustly."""
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    text = resp.choices[0].message.content.strip()
    try:
        if text.startswith("```"):
            text = text.strip("`")
            idx = text.find("{")
            if idx >= 0:
                text = text[idx:]
        return json.loads(text)
    except Exception:
        start = text.find("{"); end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise ValueError(f"Model did not return valid JSON:\n{text}")

# -----------------------------
# CSV normalization utils
# -----------------------------
def coalesce_cols(df: pd.DataFrame, names: List[str], default=None):
    for n in names:
        if n in df.columns and df[n].notna().any():
            return df[n]
    return pd.Series([default] * len(df))

def parse_choices_field(val) -> List[str]:
    """Accepts JSON array, python list repr, or delimited strings."""
    if isinstance(val, list):
        return [str(x).strip() for x in val if str(x).strip()]
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    s = str(val).strip()
    if not s:
        return []
    # Try JSON list
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    # Try python literal list
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = eval(s, {"__builtins__": {}})
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
    # Fallback: split by common delimiters
    import re
    parts = re.split(r"\s*[;|,]\s*", s)
    return [p for p in parts if p]

def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce DF with: id, question, choices(list), correct_choice, explanation
    Tries common column names automatically.
    """
    df = df.copy()
    df["question"] = coalesce_cols(df, ["question", "prompt", "stem", "text", "mcq", "MCQ"], default="").astype(str)
    df["explanation"] = coalesce_cols(df, ["explanation", "solution", "rationale", "reasoning"], default="")
    df["correct_choice"] = coalesce_cols(df, ["correct_choice", "answer", "correct", "key", "label", "gold"], default="")

    # choices column or collect option-like columns
    if "choices" in df.columns:
        df["choices"] = df["choices"].apply(parse_choices_field)
    else:
        import re
        choice_like = [c for c in df.columns
                       if re.fullmatch(r"(choice_\d+|option_\d+|opt_\d+|A|B|C|D|E|F|G|H|option[A-H])", str(c), flags=re.IGNORECASE)]
        if choice_like:
            def row_to_choices(row):
                vals = []
                for c in choice_like:
                    v = row.get(c, None)
                    if v is not None and str(v).strip():
                        vals.append(str(v).strip())
                return vals
            df["choices"] = df.apply(row_to_choices, axis=1)
        else:
            df["choices"] = coalesce_cols(df, ["options", "answers", "candidates"], default="").apply(parse_choices_field)

    if "id" not in df.columns:
        df["id"] = coalesce_cols(df, ["qid", "uid", "index", "row_id"], default=None)
        if df["id"].isna().all():
            df["id"] = [f"Q{ix+1}" for ix in range(len(df))]

    df["choices"] = df["choices"].apply(lambda x: x if isinstance(x, list) else [])
    df["correct_choice"] = df["correct_choice"].astype(str)
    df["explanation"] = df["explanation"].astype(str)
    front = ["id", "question", "choices", "correct_choice", "explanation"]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]

# -----------------------------
# Pipeline (same modular funcs)
# -----------------------------
def classify_mcq_suitability(df: pd.DataFrame,
                             question_col: str = "question",
                             model: str = MODEL) -> pd.DataFrame:
    decisions, reasons = [], []
    for q in df[question_col].tolist():
        prompt = f"{CLASSIFIER_PROMPT}\n\nMCQ:\n{q}"
        out = chat_json(prompt, model=model, temperature=0.0, max_tokens=200)
        decisions.append(out.get("decision", "No"))
        reasons.append(out.get("reason", ""))
    df = df.copy()
    df["osq_suitable"] = decisions
    df["osq_suitable_reason"] = reasons
    return df

def convert_mcq_row(row: pd.Series,
                    model: str = MODEL,
                    temperature: float = 0.2,
                    max_tokens: int = 900) -> Dict[str, Any]:
    q = row.get("question", "")
    choices = row.get("choices", [])
    correct_choice = row.get("correct_choice", "")
    explanation = row.get("explanation", "") or "N/A"
    prompt = (
        f"{CONVERTER_HEADER}\n\n"
        f"Input MCQ:\n"
        f"- Stem: {q}\n"
        f"- Choices: {json.dumps(choices, ensure_ascii=False)}\n"
        f"- Correct choice: {correct_choice}\n"
        f"- Reference explanation (if any): {explanation}\n"
    )
    return chat_json(prompt, model=model, temperature=temperature, max_tokens=max_tokens)

def convert_mcqs_to_osq(df: pd.DataFrame,
                        model: str = MODEL,
                        only_if_suitable: bool = True) -> pd.DataFrame:
    df = df.copy()
    osq_prompt, expected, rubric, difficulty = [], [], [], []
    for _, row in df.iterrows():
        if only_if_suitable and str(row.get("osq_suitable", "No")) != "Yes":
            osq_prompt.append(None); expected.append(None); rubric.append(None); difficulty.append(None)
            continue
        out = convert_mcq_row(row, model=model)
        osq_prompt.append(out.get("osq_prompt"))
        expected.append(out.get("expected_answer"))
        rubric.append(out.get("rubric"))
        difficulty.append(out.get("difficulty"))
    df["osq_prompt"] = osq_prompt
    df["expected_answer"] = expected
    df["rubric"] = rubric
    df["difficulty"] = difficulty
    return df

# -----------------------------
# Run: load → normalize → (classify) → convert → save
# -----------------------------
df_raw = pd.read_csv(CSV_IN)
df = normalize_schema(df_raw)

if SAMPLE_N and SAMPLE_N > 0:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)

if SKIP_CLASSIFY:
    df["osq_suitable"] = "Yes"
    df["osq_suitable_reason"] = "Forced conversion (skip_classify)."
else:
    df = classify_mcq_suitability(df)

df_out = convert_mcqs_to_osq(df, only_if_suitable=not SKIP_CLASSIFY)

# Save files
df_out.to_csv(CSV_OUT, index=False)
with open(JSONL_OUT, "w", encoding="utf-8") as f:
    for _, row in df_out.iterrows():
        rec = {
            "id": row.get("id"),
            "question": row.get("question"),
            "choices": row.get("choices"),
            "correct_choice": row.get("correct_choice"),
            "explanation": row.get("explanation"),
            "osq_suitable": row.get("osq_suitable"),
            "osq_suitable_reason": row.get("osq_suitable_reason"),
            "osq_prompt": row.get("osq_prompt"),
            "expected_answer": row.get("expected_answer"),
            "rubric": row.get("rubric"),
            "difficulty": row.get("difficulty"),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Display a quick sample
print(f"Input rows: {len(df_raw)} | Processed rows: {len(df_out)}")
print("First few converted items:")
display(df_out[["id","osq_suitable","osq_prompt","expected_answer","difficulty"]].head(5))
print(f"\nSaved:\n  CSV  -> {CSV_OUT}\n  JSONL-> {JSONL_OUT}")


Input rows: 1144 | Processed rows: 5
First few converted items:


,id,osq_suitable,osq_prompt,expected_answer,difficulty
0,Q219,Yes,What is the focus of Lean Engineering in syste...,The focus of Lean Engineering in system lifecy...,medium
1,Q810,Yes,Describe the recommended approach for arrangin...,The recommended approach for arranging control...,medium
2,Q502,Yes,Explain the role of the Initial Capabilities D...,The Initial Capabilities Document (ICD) outlin...,medium
3,Q650,Yes,Describe the performance requirements for ship...,Shipboard equipment must operate without malfu...,medium
4,Q324,Yes,Describe how the interaction between an actor ...,"In SysML, the interaction between an actor and...",medium



Saved:
  CSV  -> C:\Users\rabel\Desktop\dissertation\src\phase2_conversion\sysengbench_converted.csv
  JSONL-> C:\Users\rabel\Desktop\dissertation\src\phase2_conversion\sysengbench_converted.jsonl
